In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
# 1. 데이터 로드
file_path = 'dataset/seoul_pm10.csv'

In [3]:
# 데이터 읽기
df = pd.read_csv(file_path, encoding='cp949')

In [4]:
df.head()

,date,area,pm10,pm2.5
0,2022-12-31 23:00,강남구,57.0,44.0
1,2022-12-31 23:00,강동구,68.0,55.0
2,2022-12-31 23:00,강북구,59.0,42.0
3,2022-12-31 23:00,강서구,62.0,40.0
4,2022-12-31 23:00,관악구,57.0,38.0


In [5]:
# 강남구 데이터만 필터링
target_area = '강남구'
df_gangnam = df[df['area'] == target_area].copy()

In [6]:
# 시간 순서 정렬 (시게열 예측 필수)
df_gangnam['date'] = pd.to_datetime(df_gangnam['date'])
df_gangnam = df_gangnam.sort_values('date')

# PM10 컬럼의 빈 값을 시간 흐름에 따라 부드럽게 채워줌 (보간법)
df_gangnam['pm10'] = df_gangnam['pm10'].interpolate(method='linear')

print(f'[{target_area}] 데이터 추출 완료 : 총 {len(df_gangnam)}개의 샘플')

[강남구] 데이터 추출 완료 : 총 8760개의 샘플


In [7]:
# 학습에 사용할 타겟 컬럼 선택 (PM10)
data = df_gangnam['pm10'].values.reshape(-1,1)

In [9]:
# 2. 데이터 전처리 (스케일링 및 시퀀스 생성)
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(data) 

def create_dataset(dataset, time_step=24):
    X, Y = [], []
    for i in range(len(dataset) - time_step - 1) :
        a = dataset[i:(i+time_step), 0]
        X.append(a)
        Y.append(dataset[i + time_step, 0])
    return np.array(X), np.array(Y)

In [10]:

# 하이퍼파라미터
time_step = 24 # 과거 24시간 데이터를 기반으로 예측
train_size = int(len(scaled_data)* 0.8)
test_size = len(scaled_data) - train_size

train_data = scaled_data[0:train_size, :]
test_data = scaled_data[train_size:len(scaled_data), :]

X_train, y_train = create_dataset(train_data, time_step)
X_test, y_test = create_dataset(test_data, time_step)

In [11]:
# 모델 입력 차원 변환 : [Samples, Time Steps, Features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

In [12]:
# 3. 모델 정의 및 학습 함수
def train_and_predict(model_name) :
    model = Sequential()

    # 1. RNN Layer (Feature Extraction)
    # 뉴런 수를 64로 늘려 더 많은 특징을 잡아내도록 설정
    if model_name == 'LSTM' :
        model.add(LSTM(64, return_sequences=False, input_shape=(time_step, 1)))
    elif model_name == 'GRU' :
        model.add(GRU(64, return_sequences=False, input_shape=(time_step, 1)))
    elif model_name == 'BiLSTM' :
        model.add(Bidirectional(LSTM(64, return_sequences=False, input_shape=(time_step, 1))))
    
    # RNN 층 직후 과적합 방지 및 학습 안정화
    model.add(BatchNormalization()) # 데이터 분포 정규화
    model.add(Dropout(0.3)) # 뉴런 30% 비활성화

    # 2. Dense Layers (Fully Connected) - 계층적 구조화
    # 구조 : 32 => 16 (점진적으로 줄여나가며 특징 압축)

    # Hidden Layer 1 (32)
    model.add(Dense(32, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.2)) # 20% 비활성화

    # Hidden Layer 2 (16)
    model.add(Dense(16, activation='relu'))
    # 마지막 층 전에는 보통 Dropout을 약하게 하거나 생략하기도 함

    # 3. Output Layer (예측 층)
    model.add(Dense(1)) # 회귀 예측이므로 활성화 함수 없음 (Linear)

    # 컴파일
    model.compile(optimizer='adam', loss='mean_squared_error')

    # 4. Early Stopping 설정
    # val_loss가 10번 동안 개선되지 않으면 학습 조기 종료
    # restore_best_weights=True : 학습이 멈췄을 때, 가장 성능이 좋았던 시점의 가중치로 복구
    early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1)

    print(f'\n 트레이닝 {model_name} for {target_area} 모델 시작')

    # 학습 시작
    # EarlyStopping에 의해 조기 종료될 수 있음
    model.fit(X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )

    # 예측 수행
    pred = model.predict(X_test)

    # 원래 스케일로 복원
    pred_inv = scaler.inverse_transform(pred)
    y_test_inv = scaler.inverse_transform([y_test])

    # RMSE 계산
    rmse = np.sqrt(mean_squared_error(y_test_inv[0], pred_inv[:,0]))

    return rmse, pred_inv
        


In [13]:
# 4. 성능 비교 및 시각화
models = ['LSTM', 'GRU', 'BiLSTM']
results = {}
predictions = {}

for m in models :
    rmse, pred = train_and_predict(m)
    results[m] = rmse
    predictions[m] = pred
    print(f'{m} 모델 성능 평가 완료 : RMSE = {rmse:.4f}')

d:\python_sim\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



 트레이닝 LSTM for 강남구 모델 시작
Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - loss: 0.3325 - val_loss: 0.0094
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0490 - val_loss: 0.0096
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0269 - val_loss: 0.0049
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0154 - val_loss: 0.0078
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0099 - val_loss: 0.0052
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0069 - val_loss: 0.0028
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0053 - val_loss: 0.0024
Epoch 8/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0035 - val_loss: 0.0017
Epoch 9/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0030 - val_loss: 0.0013
Epoch 10/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0023 - val_loss: 0.0018
Epoch 11/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0020 - val_loss: 0.0015
Epoch 12/20
219/21

d:\python_sim\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


219/219 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.2321 - val_loss: 0.0779
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0411 - val_loss: 0.0218
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0165 - val_loss: 0.0089
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0088 - val_loss: 0.0031
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0057 - val_loss: 0.0020
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0038 - val_loss: 0.0012
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0029 - val_loss: 0.0015
Epoch 8/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0023 - val_loss: 0.0013
Epoch 9/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0018 - val_loss: 0.0010
Epoch 10/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0016 - val_loss: 0.0024
Epoch 11/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0014 - val_loss: 8.2750e-04
Epoch 12/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms

d:\python_sim\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


219/219 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - loss: 0.1165 - val_loss: 0.0093
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0291 - val_loss: 0.0066
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0137 - val_loss: 0.0043
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0089 - val_loss: 0.0031
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0059 - val_loss: 0.0027
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0047 - val_loss: 0.0022
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0035 - val_loss: 0.0024
Epoch 8/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0028 - val_loss: 0.0019
Epoch 9/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0025 - val_loss: 0.0022
Epoch 10/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0022 - val_loss: 0.0021
Epoch 11/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0019 - val_loss: 0.0017
Epoch 12/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/st

In [14]:
# 결과 요약
best_model = min(results, key=results.get)
print('\n'+'=' * 40)
print(f'[{target_area}] 예측 성능 비교')
print(f'가장 성능이 좋은 모델 : {best_model} (RMSE : {results[best_model]:.4f})')
print('=' * 40)



[강남구] 예측 성능 비교
가장 성능이 좋은 모델 : GRU (RMSE : 9.1456)
